# A6 · A tour of scikit-learn: estimators, honest evaluation, and trust
**Series A: Machine Learning Basics for Chemists, Post 6 of 7** &nbsp;|&nbsp; companion *primer* notebook

Section 2.5 of the notes gives three scikit-learn notebooks: a short introduction (an excerpt of J. VanderPlas's *Python Data Science Handbook*), a longer tutorial (*An introduction to Machine Learning with Scikit-Learn*, G. Louppe) and the CHEM 584 notebook on **robust and calibrated estimators**. They show how to *use* the library. This primer rebuilds what the library does, **with NumPy only**: the estimator interface (`fit`, `predict`, `transform`), the data of the tutorial, k-nearest neighbours, K-fold cross-validation, grid search and **nested** cross-validation, metrics, pipelines, robust scaling and robust losses, ensembling and calibration. Part I reproduces the tutorials' numbers **exactly** and then asks what they hide: how optimistic a reported cross-validation score is, how a pipeline prevents leakage, and how much to trust a probability. Part II checks every function against scikit-learn.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in well under a minute on a CPU. **Part II** (scikit-learn, preinstalled in Colab) compares every function of Part I with the library.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | the estimator interface, from scratch; the tutorials' data |
| 2 | training error, hold-out error and cross-validation |
| 3 | metrics: confusion matrix, precision, recall, F, ROC AUC |
| 4 | hyper-parameters: the validation curve, grid search, nested cross-validation |
| 5 | how optimistic is the best score? and leakage through a pipeline |
| 6 | robustness: robust scaling, robust losses, ensembling |
| 7 | calibration |
| **Part II** | cross-check with scikit-learn |

## 0. Setup

In [ ]:
import math, time, importlib.util
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

## 1. The estimator interface, from scratch
Every scikit-learn model follows one contract. An **estimator** is built with its *hyper-parameters* (`KNN(k=5)`), learns from data with **`fit(X, y)`** (the learned state goes into attributes whose names end with an underscore, `X_`, `coef_`) and returns itself; a **predictor** adds **`predict(X)`** and **`score(X, y)`**; a **transformer** has **`fit`** and **`transform`**. Data are arrays: `X` with one row per sample and one column per feature, `y` with one entry per sample. A **pipeline** is a list of transformers followed by an estimator that is itself an estimator.
**Stop and predict:** what are the slope and intercept of the least-squares line for 50 points of $y=2x-1$ plus unit noise?

In [ ]:
class LinearRegression:
    def fit(self, X, y): A = np.c_[X, np.ones(len(X))]; w = np.linalg.lstsq(A, y, rcond=None)[0]; self.coef_, self.intercept_ = w[:-1], w[-1]; return self
    def predict(self, X): return X @ self.coef_ + self.intercept_
    def score(self, X, y): p = self.predict(X); return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()                    # R^2, the default score of a regressor
rng = np.random.RandomState(42); x = 10 * rng.rand(50); y_line = 2 * x - 1 + rng.randn(50)
lr = LinearRegression().fit(x[:, None], y_line); print(f"slope {lr.coef_[0]:.7f}, intercept {lr.intercept_:.7f} | R^2 {lr.score(x[:, None], y_line):.4f} (the short notebook prints 1.9776566 and -0.9033107)")
assert abs(lr.coef_[0] - 1.9776566) < 1e-7 and abs(lr.intercept_ + 0.9033107) < 1e-7

def make_blobs_np(n_samples=1000, centers=20, seed=123, box=(-10, 10), std=1.0):
    """Gaussian blobs, built the way scikit-learn builds them (the same random-number calls)"""
    r = np.random.RandomState(seed); c = r.uniform(box[0], box[1], size=(centers, 2)) if np.isscalar(centers) else np.array(centers, float); nc = len(c)
    per = [n_samples // nc] * nc
    for i in range(n_samples % nc): per[i] += 1
    X = np.concatenate([c[k] + r.normal(scale=std, size=(per[k], 2)) for k in range(nc)]); y = np.concatenate([[k] * per[k] for k in range(nc)])
    idx = np.arange(n_samples); r.shuffle(idx); return X[idx], y[idx]
Xr, yr = make_blobs_np(); labels = np.where(yr < 10, "r", "b"); yb = (yr < 10).astype(int)                                      # the tutorial's data: blobs 0-9 are class 'r'
print("X", Xr.shape, "y", labels.shape, "| first rows", Xr[:3].round(3).tolist(), "| first labels", labels[:5].tolist(), "(the tutorial prints [-6.453 -8.764], [0.29 0.147], [-5.184 -1.253] and r r b r b)")
assert np.allclose(Xr[0], [-6.453, -8.764], atol=1e-3) and np.allclose(Xr[1], [0.29, 0.147], atol=1e-3) and labels[:5].tolist() == ["r", "r", "b", "r", "b"]

In [ ]:
class KNN:
    """k-nearest neighbours: fit stores the data, predict votes among the k closest training points (ties go to the first class)"""
    def __init__(self, k=5): self.k = k
    def fit(self, X, y): self.X_, self.y_ = np.asarray(X, float), np.asarray(y); self.classes_ = np.unique(self.y_); return self
    def _order(self, X): d = ((X[:, None, :] - self.X_[None, :, :]) ** 2).sum(-1); return np.argsort(d, axis=1, kind="stable")
    def predict_proba(self, X):
        idx = self._order(np.asarray(X, float))[:, :self.k]; return np.stack([(self.y_[idx] == c).mean(1) for c in self.classes_], 1)
    def predict(self, X): return self.classes_[self.predict_proba(X).argmax(1)]
    def score(self, X, y): return float((self.predict(X) == y).mean())
class StandardScaler:
    def fit(self, X, y=None): self.mean_, self.scale_ = X.mean(0), X.std(0); return self
    def transform(self, X): return (X - self.mean_) / self.scale_
    def fit_transform(self, X, y=None): return self.fit(X).transform(X)
class Pipeline:
    """a chain of transformers and a final estimator, which is itself an estimator"""
    def __init__(self, steps): self.steps = steps
    def fit(self, X, y):
        for t in self.steps[:-1]: X = t.fit_transform(X, y)
        self.steps[-1].fit(X, y); return self
    def _tf(self, X):
        for t in self.steps[:-1]: X = t.transform(X)
        return X
    def predict(self, X): return self.steps[-1].predict(self._tf(X))
    def score(self, X, y): return float((self.predict(X) == y).mean())
clf = KNN(5).fit(Xr, labels); print("5-NN prediction of the first five samples:", clf.predict(Xr[:5]).tolist(), "| class probabilities (the tutorial prints [0 1], [0 1], [0.2 0.8], [0.6 0.4], [0.8 0.2] for 'b', 'r'):")
print(clf.predict_proba(Xr[:5]))
assert clf.predict(Xr[:5]).tolist() == ["r", "r", "r", "b", "b"] and np.allclose(clf.predict_proba(Xr[:5])[2], [0.2, 0.8])

## 2. Training error, hold-out error, cross-validation
The **training error** (error on the data the model was fitted on) is a biased, optimistic estimate of the error on new data: a 1-nearest-neighbour classifier remembers its training set and has training error $0$. The **hold-out** error uses data kept aside, but it depends on *which* data; **$K$-fold cross-validation** splits the data into $K$ folds, trains on $K-1$ and tests on the remaining one in turn, and averages.
**Stop and predict:** the training error of the 1-NN classifier, and how much the hold-out error changes from one random 75/25 split to the next.

In [ ]:
def kfold(n, k=5):
    sizes = np.full(k, n // k); sizes[: n % k] += 1; cur = 0
    for s in sizes: te = np.arange(cur, cur + s); tr = np.r_[np.arange(0, cur), np.arange(cur + s, n)]; cur += s; yield tr, te
def cv_scores(make, X, y, splits): return np.array([make().fit(X[tr], y[tr]).score(X[te], y[te]) for tr, te in splits])
print("training error of 1-NN:", 1 - KNN(1).fit(Xr, yb).score(Xr, yb), "(the tutorial prints 0.0)")
sc = cv_scores(lambda: KNN(5), Xr, yb, kfold(1000, 5)); print(f"5-fold cross-validation error of 5-NN: {1 - sc.mean():.6f} +- {sc.std():.6f} (the tutorial prints 0.163000 +-0.010770)")
rg = np.random.default_rng(0); te_err, tr_err = [], []
for _ in range(300):
    p_ = rg.permutation(1000); tr, te = p_[:750], p_[750:]; m = KNN(5).fit(Xr[tr], yb[tr]); tr_err.append(1 - m.score(Xr[tr], yb[tr])); te_err.append(1 - m.score(Xr[te], yb[te]))
print(f"300 random 75/25 splits: training error {np.mean(tr_err):.3f}, test error {np.mean(te_err):.3f} +- {np.std(te_err):.3f} (range {np.min(te_err):.3f} to {np.max(te_err):.3f}) | the tutorial's single split gives 0.107 and 0.184")
assert abs((1 - sc.mean()) - 0.163) < 1e-9 and abs(sc.std() - 0.010770) < 1e-6 and np.mean(tr_err) < np.mean(te_err) and np.std(te_err) > 0.015

## 3. Metrics
For a binary problem with *positive* class `r`, a **confusion matrix** counts the true classes in the rows and the predicted classes in the columns; **accuracy** $=(TP+TN)/N$, **precision** $=TP/(TP+FP)$, **recall** $=TP/(TP+FN)$ and the **F measure** $=2TP/(2TP+FP+FN)$. **ROC AUC** is the area under the curve of the true-positive rate against the false-positive rate as the decision threshold moves; it equals the probability that a random positive sample gets a higher score than a random negative one. The tutorial prints a confusion matrix; we recompute its other numbers from it.
**Stop and predict:** for $\begin{pmatrix}86&19\\13&82\end{pmatrix}$, which entry is TP, and what are precision and recall?

In [ ]:
cm = np.array([[86, 19], [13, 82]]); tn, fp, fn, tp = cm.ravel()
acc, prec, rec = (tp + tn) / cm.sum(), tp / (tp + fp), tp / (tp + fn); f1 = 2 * tp / (2 * tp + fp + fn)
print(f"the tutorial's confusion matrix sums to {cm.sum()} test samples: accuracy {acc:.4f}, precision {prec:.10f}, recall {rec:.10f}, F {f1:.10f}")
print("the tutorial prints 0.84, 0.8118811881188119, 0.8631578947368421, 0.8367346938775511: all four follow from the matrix; but its 'Test error' output (0.184 = 46 errors) belongs to a test set of", int(np.ceil(0.25 * 1000)), "samples (75/25 of 1000), so the two outputs come from different splits")
assert abs(acc - 0.84) < 1e-12 and abs(prec - 0.8118811881188119) < 1e-15 and abs(rec - 0.8631578947368421) < 1e-15 and abs(f1 - 0.8367346938775511) < 1e-15
def roc_auc(y, s):
    """AUC as the probability that a positive sample outranks a negative one (ties count 1/2)"""
    pos, neg = s[y == 1], s[y == 0]; return float(((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg)))
def roc_curve_auc(y, s):
    """the ROC curve at the DISTINCT score values (tied scores share one threshold), and its area by the trapezoid rule"""
    order = np.argsort(-s, kind="stable"); ss, ys = s[order], y[order]; last = np.r_[np.where(np.diff(ss) != 0)[0], len(ss) - 1]
    tpr = np.r_[0, np.cumsum(ys)[last] / ys.sum()]; fpr = np.r_[0, np.cumsum(1 - ys)[last] / (1 - ys).sum()]; return fpr, tpr, float(((fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) / 2).sum())
p_ = np.random.default_rng(1).permutation(1000); tr, te = p_[:750], p_[750:]; m = KNN(5).fit(Xr[tr], yb[tr]); score_te = m.predict_proba(Xr[te])[:, 1]
y_auc = yb[te]; fpr, tpr, auc_curve = roc_curve_auc(y_auc, score_te); auc_rank = roc_auc(y_auc, score_te)
print(f"on one split of our own: AUC from the ranks {auc_rank:.6f}, from the area under the ROC curve {auc_curve:.6f} (the same number, in two ways; the k-NN scores are multiples of 0.2, so many are tied)")
assert abs(auc_rank - auc_curve) < 1e-12

## 4. Hyper-parameters: the validation curve, grid search, nested cross-validation
The number of neighbours $k$ is a **hyper-parameter**: small $k$ over-fits (low training error, high test error), large $k$ under-fits. A **grid search** evaluates every value by cross-validation and keeps the best. Two details follow scikit-learn: for a classifier the folds are **stratified** (each fold keeps the class proportions), and the reported `best_score_` is the cross-validation score of the winner.
**Stop and predict:** is `best_score_` an unbiased estimate of the error of the selected model on new data?

In [ ]:
def stratified_kfold(y, k=5):
    y = np.asarray(y); classes = np.unique(y); enc = np.searchsorted(classes, y); yo = np.sort(enc)
    alloc = np.array([np.bincount(yo[i::k], minlength=len(classes)) for i in range(k)]); folds = np.zeros(len(y), int)
    for c in range(len(classes)): folds[enc == c] = np.arange(k).repeat(alloc[:, c])
    for f in range(k): yield np.where(folds != f)[0], np.where(folds == f)[0]
def knn_fold_scores(Xtr, ytr, Xte, yte, ks):
    """accuracy of k-NN for every k at once: sort the neighbours once, then count class-1 votes with a cumulative sum"""
    d = ((Xte[:, None, :] - Xtr[None, :, :]) ** 2).sum(-1); order = np.argsort(d, axis=1, kind="stable"); csum = np.cumsum(ytr[order], axis=1)
    return np.array([((csum[:, k - 1] > k - csum[:, k - 1]).astype(int) == yte).mean() for k in ks])                       # ties go to class 0
def grid_search_knn(X, y, ks, splits):
    s = np.mean([knn_fold_scores(X[tr], y[tr], X[te], y[te], ks) for tr, te in splits], axis=0); b = int(np.argmax(s)); return ks[b], s[b], s
ks = list(range(1, 100)); splits = list(stratified_kfold(yb, 5)); k_best, s_best, s_all = grid_search_knn(Xr, yb, ks, splits)
train_err = np.array([1 - KNN(k).fit(Xr, yb).score(Xr, yb) for k in (1, 2, 5, 10, 34, 60, 99)])
print(f"grid search over k = 1 ... 99 with stratified 5-fold cross-validation: best k = {k_best}, cross-validation error {1 - s_best:.6f} (the tutorial: k = 34, 0.131000)")
print("k:                  1      2      5     10     34     60     99\ntraining error: ", np.round(train_err, 3), "\nCV error:       ", np.round(1 - s_all[[0, 1, 4, 9, 33, 59, 98]], 3))
nested = []
for tr, te in stratified_kfold(yb, 5):                                                                                       # the outer loop: the winner of the inner grid search is tested on data that did not take part in the search
    kk, _, _ = grid_search_knn(Xr[tr], yb[tr], ks, list(stratified_kfold(yb[tr], 5))); nested.append(knn_fold_scores(Xr[tr], yb[tr], Xr[te], yb[te], [kk])[0])
nested = np.array(nested); print(f"nested cross-validation: error {1 - nested.mean():.6f} +- {nested.std():.6f} (the tutorial: 0.144000 +-0.023958) | the grid search's own number was {1 - s_best:.3f}: optimistic by {(1 - nested.mean()) - (1 - s_best):.3f}")
assert k_best == 34 and abs((1 - s_best) - 0.131) < 1e-9 and abs((1 - nested.mean()) - 0.144) < 1e-9 and abs(nested.std() - 0.023958) < 1e-6

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.8)); kk_ = np.arange(1, 100); tr_curve = np.array([1 - KNN(k).fit(Xr, yb).score(Xr, yb) for k in range(1, 100, 3)])
ax.plot(range(1, 100, 3), tr_curve, "o-", ms=3, color=COLORS["orange"], label="training error"); ax.plot(kk_, 1 - s_all, color=COLORS["blue"], label="cross-validation error"); ax.axvline(k_best, color="k", ls="--", lw=1, label=f"best k = {k_best}")
ax.set(xlabel="number of neighbours k", ylabel="error", title="Under- and over-fitting of k-NN"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 5. How optimistic is the best score, and what is leakage?
**(a)** The winner of a search over 99 values of $k$ is *chosen because* its cross-validation score is the best: part of that score is luck of the folds. We measure the optimism: for datasets of 100, 300 and 1,000 samples (the same blob recipe), compare the best cross-validation error with the error of the selected $k$ on 5,000 independent samples. As the extreme case, labels with **no signal at all**.
**(b)** **Leakage**: any step that uses the labels (scaling does not, feature selection does) must be fitted inside each training fold. We choose the 10 features most correlated with the label among 2,000 pure-noise features, with and without putting the selection in the pipeline.
**Stop and predict:** what accuracy should a classifier get on noise labels, and what does cross-validation report if the selection is done on all the data?

In [ ]:
def optimism(n, seed, noise_labels=False, n_test=5000):
    X, y = make_blobs_np(n + n_test, 20, seed=seed); y = (y < 10).astype(int); Xd, yd, Xt, yt = X[:n], y[:n], X[n:], y[n:]
    if noise_labels: yd = np.random.default_rng(seed).integers(0, 2, n); yt = np.random.default_rng(seed + 1).integers(0, 2, n_test)
    kmax = min(99, int(0.8 * n) - 1); kl = list(range(1, kmax + 1)); kb, sb, _ = grid_search_knn(Xd, yd, kl, list(stratified_kfold(yd, 5)))
    return 1 - sb, 1 - knn_fold_scores(Xd, yd, Xt, yt, [kb])[0]
rows = []
for n in (100, 300, 1000):
    r_ = np.array([optimism(n, s) for s in range(1000, 1030)]); rows.append((f"blobs, n = {n}", r_[:, 0].mean(), r_[:, 1].mean(), (r_[:, 1] - r_[:, 0]).mean(), (r_[:, 1] - r_[:, 0]).std() / math.sqrt(len(r_))))
r_ = np.array([optimism(100, s, True) for s in range(1000, 1030)]); rows.append(("NO signal (random labels), n = 100", r_[:, 0].mean(), r_[:, 1].mean(), (r_[:, 1] - r_[:, 0]).mean(), (r_[:, 1] - r_[:, 0]).std() / math.sqrt(len(r_))))
opt = pd.DataFrame(rows, columns=["dataset (30 repeats)", "best CV error", "error on 5,000 new samples", "optimism", "standard error"]); print(opt.round(3).to_string(index=False))
assert all(opt["optimism"] > 0) and opt.iloc[3, 3] > 0.05 and abs(opt.iloc[3, 2] - 0.5) < 0.03

In [ ]:
def leak_experiment(seed, n=60, p=2000, k_feat=10):
    r = np.random.default_rng(seed); X = r.normal(size=(n, p)); y = r.integers(0, 2, n)
    def select(Xa, ya):
        d = np.abs(Xa[ya == 1].mean(0) - Xa[ya == 0].mean(0)) / (Xa.std(0) + 1e-12); return np.argsort(-d)[:k_feat]
    def centroid_acc(Xtr, ytr, Xte, yte):
        c0, c1 = Xtr[ytr == 0].mean(0), Xtr[ytr == 1].mean(0); pred = (((Xte - c1) ** 2).sum(1) < ((Xte - c0) ** 2).sum(1)).astype(int); return (pred == yte).mean()
    sel_all = select(X, y); folds = list(stratified_kfold(y, 5))
    leaky = np.mean([centroid_acc(X[tr][:, sel_all], y[tr], X[te][:, sel_all], y[te]) for tr, te in folds])             # the selection saw the labels of the test folds
    proper = np.mean([centroid_acc(X[tr][:, (s := select(X[tr], y[tr]))], y[tr], X[te][:, s], y[te]) for tr, te in folds])                 # the selection is part of each training fold
    return leaky, proper
res = np.array([leak_experiment(s) for s in range(50)])
print(f"60 samples, 2,000 noise features, labels at random: cross-validated accuracy with the selection done on ALL the data {res[:, 0].mean():.3f} +- {res[:, 0].std():.3f}; with the selection inside every fold {res[:, 1].mean():.3f} +- {res[:, 1].std():.3f} (50 datasets; the truth is 0.5)")
assert res[:, 0].mean() > 0.75 and abs(res[:, 1].mean() - 0.5) < 0.03

## 6. Robustness: scaling, losses, ensembling
**(a) Scaling with an outlier.** Standardization uses the mean and the standard deviation, which one extreme value can ruin; **robust scaling** uses the median and the interquartile range. The notebook puts one outlier ($-1000$) in the training set of a 2D classification problem and fits a 5-NN classifier.
**(b) The loss.** Least squares minimizes $\sum r_i^2$, which gives large residuals great weight; the least absolute deviation $\sum|r_i|$ does not.
**(c) Ensembling.** The expected squared error splits into noise, bias$^2$ and **variance**; averaging many models fitted to resampled data reduces the variance.
**Stop and predict:** accuracy with the standard and with the robust scaler?

In [ ]:
Xs_, ys_ = make_blobs_np(100, [(0, 0), (-1, 0)], seed=0); perm = np.random.RandomState(0).permutation(100); te, tr = perm[:50], perm[50:]            # train_test_split(test_size=0.5, random_state=0)
Xtr_, Xte_, ytr_, yte_ = Xs_[tr].copy(), Xs_[te], ys_[tr], ys_[te]; Xtr_[0, 0] = -1000                                                                  # one large outlier
class RobustScaler:
    def fit(self, X, y=None): self.center_ = np.median(X, 0); q = np.percentile(X, [25, 75], axis=0); self.scale_ = q[1] - q[0]; return self
    def transform(self, X): return (X - self.center_) / self.scale_
    def fit_transform(self, X, y=None): return self.fit(X).transform(X)
a_std = Pipeline([StandardScaler(), KNN(5)]).fit(Xtr_, ytr_).score(Xte_, yte_); a_rob = Pipeline([RobustScaler(), KNN(5)]).fit(Xtr_, ytr_).score(Xte_, yte_)
print(f"test accuracy with the standard scaler {a_std:.3f}, with the robust scaler {a_rob:.3f} (the notebook prints 0.460 and 0.680)")
print("why: the standard deviation of the first feature is", round(float(Xtr_[:, 0].std()), 1), "with the outlier against", round(float(Xtr_[1:, 0].std()), 2), "without it")
assert abs(a_std - 0.46) < 1e-9 and abs(a_rob - 0.68) < 1e-9

In [ ]:
r_ = np.random.default_rng(0); xs = r_.normal(size=100); ys = 3 * xs + r_.normal(0, 10 * 0.1, 100); xs[-3:] = 1 + 0.25 * r_.normal(size=3); ys[-3:] = -100 + 10 * r_.normal(size=3)       # 3 outliers
A = np.c_[xs, np.ones(100)]; w_ls = np.linalg.lstsq(A, ys, rcond=None)[0]
w = w_ls.copy()
for _ in range(200):                                                                                                         # least absolute deviation by iteratively reweighted least squares
    sw = 1 / np.maximum(np.abs(ys - A @ w), 1e-6); w = np.linalg.solve(A.T @ (A * sw[:, None]), A.T @ (sw * ys))
w_clean = np.linalg.lstsq(A[:-3], ys[:-3], rcond=None)[0]
print(f"line fitted to 100 points whose slope is 3, with 3 outliers: least squares slope {w_ls[0]:.2f}, least absolute deviation slope {w[0]:.2f}, least squares without the outliers {w_clean[0]:.2f}")
assert abs(w[0] - 3) < abs(w_ls[0] - 3) and abs(w[0] - w_clean[0]) < 0.3
# bagging reduces variance: 1-nearest-neighbour regression (low bias, high variance) on noisy samples of sin(2 pi x); the prediction at x0 = 0.3 over 300 training sets
def nn1_predict(xx, yy, x0): return yy[np.argmin(np.abs(xx - x0))]
def predict_single_and_bagged(seed, x0=0.3, n=30, n_bags=25):
    r = np.random.default_rng(seed); xx = r.uniform(0, 1, n); yy = np.sin(2 * np.pi * xx) + r.normal(0, 0.3, n); single = nn1_predict(xx, yy, x0); preds = []
    for _ in range(n_bags): b = r.integers(0, n, n); preds.append(nn1_predict(xx[b], yy[b], x0))
    return single, np.mean(preds)
P_ = np.array([predict_single_and_bagged(s) for s in range(1000)]); truth = np.sin(2 * np.pi * 0.3)
v1, v2 = P_[:, 0].var(), P_[:, 1].var()
print(f"prediction at x = 0.3 (truth {truth:.3f}) over 1000 training sets: a single 1-NN fit has variance {v1:.3f} and bias {P_[:, 0].mean() - truth:+.3f}; the average of 25 bootstrap fits has variance {v2:.3f} and bias {P_[:, 1].mean() - truth:+.3f}: the variance is reduced {v1 / v2:.1f} times")
print(f"expected squared error = noise + bias^2 + variance: single {0.09 + (P_[:, 0].mean() - truth) ** 2 + v1:.3f}, bagged {0.09 + (P_[:, 1].mean() - truth) ** 2 + v2:.3f} (noise variance 0.3^2 = 0.09)")
assert v2 < v1 and abs(P_[:, 1].mean() - truth) < 0.1

## 7. Calibration
A classifier that says "0.8" should be right about $80\%$ of the time for such predictions. A **reliability** check bins the predicted probabilities and compares each bin's mean prediction with the observed frequency of the positive class; the **Brier score** is $\frac1N\sum(p_i-y_i)^2$ and the **expected calibration error** is $\sum_b\frac{n_b}N|\bar y_b-\bar p_b|$. Gaussian naive Bayes, which models each class by a Gaussian per feature, is a classic miscalibrated model. The notebook repairs it with **isotonic regression**, a monotone step function fitted by the pool-adjacent-violators algorithm, trained on held-out predictions (`CalibratedClassifierCV` with 2 folds).
**Stop and predict:** the data are three blobs, the middle one half class 0 and half class 1. What probability should the best model give in the middle blob?

In [ ]:
class GaussianNB:
    def fit(self, X, y):
        self.classes_ = np.unique(y); eps = 1e-9 * X.var(0).max(); self.theta_ = np.array([X[y == c].mean(0) for c in self.classes_]); self.var_ = np.array([X[y == c].var(0) + eps for c in self.classes_]); self.prior_ = np.array([(y == c).mean() for c in self.classes_]); return self
    def predict_proba(self, X):
        jll = np.stack([np.log(self.prior_[i]) - 0.5 * np.log(2 * np.pi * self.var_[i]).sum() - 0.5 * (((X - self.theta_[i]) ** 2) / self.var_[i]).sum(1) for i in range(len(self.classes_))], 1)
        jll -= jll.max(1, keepdims=True); e = np.exp(jll); return e / e.sum(1, keepdims=True)
def isotonic_fit(x, y):
    """pool adjacent violators: the nondecreasing step function closest to y in least squares; returns (thresholds, values)"""
    ux, inv = np.unique(x, return_inverse=True); wsum = np.bincount(inv).astype(float); ysum = np.bincount(inv, weights=y); blocks = []
    for i in range(len(ux)):
        blocks.append([ysum[i] / wsum[i], wsum[i], i, i])
        while len(blocks) > 1 and blocks[-2][0] >= blocks[-1][0]:
            b2, b1 = blocks.pop(), blocks.pop(); w_ = b1[1] + b2[1]; blocks.append([(b1[0] * b1[1] + b2[0] * b2[1]) / w_, w_, b1[2], b2[3]])
    fitted = np.zeros(len(ux))
    for v, w_, s, e in blocks: fitted[s:e + 1] = v
    return ux, fitted
isotonic_predict = lambda th, vals, x: np.interp(x, th, vals)
Xc, yc = make_blobs_np(10000, [(-5, -5), (0, 0), (5, 5)], seed=0); Xc, yc = Xc, None
r0 = np.random.RandomState(0)
# the notebook's construction: blobs generated WITHOUT shuffling; the first half of the samples is class 0, the second half class 1
cen = np.array([(-5, -5), (0, 0), (5, 5)], float); per = [3334, 3333, 3333]; Xc = np.concatenate([cen[k] + r0.normal(size=(per[k], 2)) for k in range(3)]); yc = np.r_[np.zeros(5000, int), np.ones(5000, int)]
perm = np.random.RandomState(42).permutation(10000); te_c, tr_c = perm[:5000], perm[5000:]; Xa, ya, Xb_, yb_ = Xc[tr_c], yc[tr_c], Xc[te_c], yc[te_c]
gnb = GaussianNB().fit(Xa, ya); p_raw = gnb.predict_proba(Xb_)[:, 1]
cal = np.zeros(len(Xb_))
for fa, fb in stratified_kfold(ya, 2):                                                                                       # CalibratedClassifierCV(cv=2, method="isotonic"): fit on one fold, calibrate on the other, average
    m_ = GaussianNB().fit(Xa[fa], ya[fa]); th, vals = isotonic_fit(m_.predict_proba(Xa[fb])[:, 1], ya[fb].astype(float)); cal += isotonic_predict(th, vals, m_.predict_proba(Xb_)[:, 1]) / 2
brier = lambda p, y: float(((p - y) ** 2).mean())
def ece(p, y, bins=10):
    b = np.minimum((p * bins).astype(int), bins - 1); return float(sum((b == i).mean() * abs(y[b == i].mean() - p[b == i].mean()) for i in range(bins) if (b == i).any()))
mid = (np.linalg.norm(Xb_ - [0, 0], axis=1) < 1.5)
print(pd.DataFrame({"": ["Gaussian naive Bayes", "isotonic calibration"], "Brier score": [brier(p_raw, yb_), brier(cal, yb_)], "expected calibration error": [ece(p_raw, yb_), ece(cal, yb_)], "mean P(y=1) in the middle blob": [p_raw[mid].mean(), cal[mid].mean()]}).round(4).to_string(index=False))
spread = lambda p: np.percentile(p[mid], [5, 95])
print(f"observed frequency of class 1 in the middle blob: {yb_[mid].mean():.3f} (the best possible probability there is 0.5); 5th to 95th percentile of the predicted probability of the middle-blob points: naive Bayes {spread(p_raw)[0]:.2f} to {spread(p_raw)[1]:.2f}, calibrated {spread(cal)[0]:.2f} to {spread(cal)[1]:.2f}")
bins_ = np.minimum((p_raw * 10).astype(int), 9)
print("naive Bayes, the lowest bins of the reliability table (mean prediction, observed frequency of class 1, points): " + "; ".join(f"{p_raw[bins_ == i].mean():.3f}, {yb_[bins_ == i].mean():.3f}, {(bins_ == i).sum()}" for i in (0, 1)))
assert brier(cal, yb_) < brier(p_raw, yb_) and ece(cal, yb_) < ece(p_raw, yb_) and np.ptp(spread(p_raw)) > 0.4 and np.ptp(spread(cal)) < 0.2 and yb_[bins_ == 1].mean() > 0.4 and p_raw[bins_ == 0].mean() < 0.02

In [ ]:
def reliability(p, y, bins=10):
    b = np.minimum((p * bins).astype(int), bins - 1); return np.array([(p[b == i].mean(), y[b == i].mean(), (b == i).sum()) for i in range(bins) if (b == i).any()])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.9)); ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfectly calibrated")
for lab, p_, c_ in (("naive Bayes", p_raw, COLORS["red"]), ("isotonic calibration", cal, COLORS["blue"])): rr_ = reliability(p_, yb_); ax[0].plot(rr_[:, 0], rr_[:, 1], "o-", ms=4, color=c_, label=lab)
ax[0].set(xlabel="mean predicted probability in the bin", ylabel="observed frequency of class 1", title="(a) reliability diagram"); ax[0].legend(fontsize=8)
ax[1].hist(p_raw[mid], bins=20, range=(0, 1), color=COLORS["red"], alpha=0.6, label="naive Bayes"); ax[1].hist(cal[mid], bins=20, range=(0, 1), color=COLORS["blue"], alpha=0.6, label="isotonic calibration")
ax[1].set(xlabel="predicted P(y = 1)", ylabel="points of the middle blob", title="(b) the middle blob, where the truth is 0.5"); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

## 8. Your turn
**Exercise 1.** Run the grid search with 3-fold and 10-fold cross-validation. How do the best $k$, its score and the nested score change?
**Exercise 2.** Increase the number of noise features in Section 5(b) to 20,000, or decrease the number of selected features to 2. How does the leaky accuracy change?
**Exercise 3.** Add the same outlier ($-1000$) at a test point instead of a training point. Which scaler suffers?
**Exercise 4.** Fit the calibrated model with 5 folds instead of 2. Does the Brier score improve?
**Exercise 5.** Replace the k-NN by a classifier of your own with `fit` and `predict`, and put it in the pipeline of Section 1.

## 9. Self-check

In [ ]:
assert abs(lr.coef_[0] - 1.9776566) < 1e-7 and k_best == 34 and abs((1 - nested.mean()) - 0.144) < 1e-9 and abs(sc.std() - 0.010770) < 1e-6
assert abs(a_std - 0.46) < 1e-9 and abs(a_rob - 0.68) < 1e-9 and abs(f1 - 0.8367346938775511) < 1e-15 and res[:, 0].mean() > 0.75 and brier(cal, yb_) < brier(p_raw, yb_)
print("All checks passed: you reproduced the numbers in the post.")

# Part II. Cross-check with scikit-learn
> **Read this first.** Part II compares every function of Part I with scikit-learn (preinstalled in Colab): the data generator, k-NN, the K-fold and stratified folds, cross-validation, grid search and nested cross-validation, the robust scaler, Gaussian naive Bayes, isotonic calibration, ROC AUC and the Brier score, and then reproduces the numbers that the tutorials print with the library (including the digits example, where scaling *lowers* the accuracy of the SVM). Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
HAVE_SK = importlib.util.find_spec("sklearn") is not None; checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if HAVE_SK:
    import sklearn, warnings; warnings.filterwarnings("ignore")
    from sklearn.datasets import make_blobs, load_digits, load_iris
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, GridSearchCV, train_test_split
    from sklearn.preprocessing import StandardScaler as SkStandard, RobustScaler as SkRobust
    from sklearn.pipeline import make_pipeline
    from sklearn.naive_bayes import GaussianNB as SkGNB
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix
    from sklearn.svm import SVC
    print("scikit-learn", sklearn.__version__)
else: print("scikit-learn is not available: install it (pip install scikit-learn) and re-run this cell.")

In [ ]:
if HAVE_SK:
    try:
        Xs, ys = make_blobs(n_samples=1000, centers=20, random_state=123); check("make_blobs: our generator equals scikit-learn's (X and y)", np.allclose(Xs, Xr) and np.array_equal(ys, yr))
        f_sk = list(KFold(5).split(Xr)); f_me = list(kfold(1000, 5)); check("K-fold: the same folds", all(np.array_equal(a[1], b[1]) for a, b in zip(f_sk, f_me)))
        g_sk = list(StratifiedKFold(5).split(Xr, yb)); g_me = list(stratified_kfold(yb, 5)); check("stratified K-fold: the same folds", all(np.array_equal(a[1], b[1]) for a, b in zip(g_sk, g_me)))
        sk_knn = KNeighborsClassifier(5).fit(Xr[:800], yb[:800]); check("5-NN: the same predictions and class probabilities on 200 samples", np.array_equal(sk_knn.predict(Xr[800:]), KNN(5).fit(Xr[:800], yb[:800]).predict(Xr[800:])) and np.allclose(sk_knn.predict_proba(Xr[800:]), KNN(5).fit(Xr[:800], yb[:800]).predict_proba(Xr[800:])))
        s_sk = cross_val_score(KNeighborsClassifier(5), Xr, yb, cv=KFold(5), scoring="accuracy"); check(f"cross-validation error {1 - s_sk.mean():.6f} +- {s_sk.std():.6f} equals the tutorial's 0.163000 +-0.010770", abs((1 - s_sk.mean()) - 0.163) < 1e-9 and abs(s_sk.std() - 0.010770) < 1e-6)
        gs = GridSearchCV(KNeighborsClassifier(), param_grid={"n_neighbors": ks}, scoring="accuracy", cv=5).fit(Xr, yb); check(f"grid search: best k = {gs.best_params_['n_neighbors']} with error {1 - gs.best_score_:.6f}, equal to ours ({k_best}, {1 - s_best:.6f})", gs.best_params_["n_neighbors"] == k_best and abs(gs.best_score_ - s_best) < 1e-12)
        ns = cross_val_score(GridSearchCV(KNeighborsClassifier(), param_grid={"n_neighbors": ks}, scoring="accuracy", cv=5), Xr, yb, cv=5, scoring="accuracy"); check(f"nested cross-validation {1 - ns.mean():.6f} +- {ns.std():.6f} equals ours", abs(ns.mean() - nested.mean()) < 1e-12 and abs(ns.std() - nested.std()) < 1e-12)
    except Exception as e: print("stage could not be completed:", type(e).__name__, e)

In [ ]:
if HAVE_SK:
    try:
        Xs2, ys2 = make_blobs(n_samples=100, centers=[(0, 0), (-1, 0)], random_state=0); a_, b_, c_, d_ = train_test_split(Xs2, ys2, test_size=0.5, random_state=0); a_[0, 0] = -1000
        acc_sk = [make_pipeline(S, KNeighborsClassifier()).fit(a_, c_).score(b_, d_) for S in (SkStandard(), SkRobust())]; check(f"scaling with an outlier: scikit-learn gives {acc_sk[0]:.3f} and {acc_sk[1]:.3f}, the notebook 0.460 and 0.680, ours {a_std:.3f} and {a_rob:.3f}", np.allclose(acc_sk, [0.46, 0.68]) and np.allclose(acc_sk, [a_std, a_rob]))
        gnb_sk = SkGNB().fit(Xa, ya); check("Gaussian naive Bayes: the same class probabilities (largest difference below 1e-10)", np.abs(gnb_sk.predict_proba(Xb_)[:, 1] - p_raw).max() < 1e-10)
        cal_sk = CalibratedClassifierCV(SkGNB(), cv=2, method="isotonic").fit(Xa, ya).predict_proba(Xb_)[:, 1]; d_cal = np.abs(cal_sk - cal).max(); check(f"isotonic calibration (2 folds): our probabilities equal CalibratedClassifierCV's (largest difference {d_cal:.1e})", d_cal < 1e-8)
        check("Brier score equals brier_score_loss", abs(brier(cal, yb_) - brier_score_loss(yb_, cal)) < 1e-12)
        check("ROC AUC equals roc_auc_score", abs(auc_rank - roc_auc_score(y_auc, score_te)) < 1e-12)
        check("confusion matrix: rows are the true classes (scikit-learn's convention, the one used for the tutorial's matrix)", np.array_equal(confusion_matrix([0, 0, 1, 1, 1], [0, 1, 1, 1, 0]), np.array([[1, 1], [1, 2]])))
    except Exception as e: print("stage could not be completed:", type(e).__name__, e)

In [ ]:
if HAVE_SK:
    try:
        d = load_digits(); Xd, yd = d.data, d.target; Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(Xd, yd, random_state=0); tf = SkStandard().fit(Xtr_d)
        a0 = SVC().fit(Xtr_d, ytr_d).score(Xte_d, yte_d); a1 = SVC().fit(tf.transform(Xtr_d), ytr_d).score(tf.transform(Xte_d), yte_d)
        check(f"digits, SVM: accuracy {a0:.4f} without scaling and {a1:.4f} with scaling (the tutorial prints 0.9911 and 0.9844): scaling LOWERS it here", abs(a0 - 0.9911) < 2e-3 and abs(a1 - 0.9844) < 2e-3 and a1 < a0)
        Xi, yi = load_iris(return_X_y=True); a_i, b_i, c_i, d_i = train_test_split(Xi, yi, random_state=1); acc_i = SkGNB().fit(a_i, c_i).score(b_i, d_i); check(f"iris, Gaussian naive Bayes: accuracy {acc_i:.7f} (the short notebook prints 0.9736842)", abs(acc_i - 0.9736842) < 1e-6)
        a_d, b_d, c_d, d_d = train_test_split(Xd, yd, random_state=0); acc_d = SkGNB().fit(a_d, c_d).score(b_d, d_d); check(f"digits, Gaussian naive Bayes: accuracy {acc_d:.7f} (the short notebook prints 0.8333333)", abs(acc_d - 0.8333333) < 1e-6)
    except Exception as e: print("stage could not be completed:", type(e).__name__, e)
    ok = [lab for lab, p in checks if p]; bad = [lab for lab, p in checks if not p]; print(f"\nPart II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad: print("  not passed:", lab)

### What to look for
* **Exact agreement.** The generator, the folds, k-NN, the grid search and the nested scores of Part I are *identical* to scikit-learn's, not merely close: a from-scratch version of the tutorial's pipeline gives the tutorial's numbers.
* **Selection optimism.** The grid search's own score (0.131) is lower than the nested score (0.144): the winner is tested on the data that chose it.
* **Scaling is not always a gain.** On digits the SVM is slightly *better* without scaling (0.9911 against 0.9844), although the tutorial says scaling is critical for some algorithms; it matters for k-NN with an outlier (0.46 against 0.68).
* **Calibration.** Gaussian naive Bayes is repaired by isotonic regression fitted on held-out predictions; the Brier score and the expected calibration error both fall.